In [1]:
import mysql.connector
from credenciales import mysql_config

#### Creación de tablas en SQL
```sql
Tabla ESYRCE 
CREATE TABLE ESYRCE (
    Categoria   VARCHAR(100),
    Cultivo     VARCHAR(100),
    Provincia   VARCHAR(100),
    Anio        INT,
    Hectareas   FLOAT,
    PRIMARY KEY (Cultivo, Provincia, Anio)
);

Tabla AEMET
CREATE TABLE AEMET (
    Anio        INT,
    Temp_media  FLOAT,
    Temp_min    FLOAT,
    Temp_max    FLOAT,
    Precipitacion   FLOAT,
    Provincia   VARCHAR(100),
    PRIMARY KEY (Provincia, Anio)
);

Tabla NASA
CREATE TABLE NASA (
    YEAR        INT,
    PRECTOTCORR FLOAT,
    RH2M        FLOAT,
    T2M         FLOAT,
    T2M_MAX     FLOAT,
    T2M_MIN     FLOAT,
    PRIMARY KEY (YEAR)
);

Tabla FAOSTAT
CREATE TABLE FAOSTAT(
    Area        VARCHAR(100),
    Elemento    VARCHAR(100),
    Producto    VARCHAR(100),
    Anio        INT,
    Unidad      VARCHAR(100),
    Valor       FLOAT,
    Simbolo     VARCHAR(100),
    Descripción_Simbolo     VARCHAR(100),
    PRIMARY KEY (Producto, Elemento, Anio)
);
```

In [2]:
#Función para crear las tablas en TiDB.
def cargar_csv_en_tabla(df, tabla, conn, batch=1000):
    """Inserta un DataFrame en una tabla de TiDB en lotes de `batch` filas."""
    cursor = conn.cursor()
    cols = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO {tabla} ({cols}) VALUES ({placeholders})"

    filas = [tuple(None if pd.isna(v) else v for v in row)
             for row in df.itertuples(index=False, name=None)]

    for i in range(0, len(filas), batch):
        cursor.executemany(sql, filas[i:i+batch])
        conn.commit()
        print(f"  insertadas {min(i+batch, len(filas))}/{len(filas)}")
    cursor.close()
conn = mysql.connector.connect(**mysql_config)